# EDA 01 — Orders table (late delivery focus)

Source: `olist.bronze.orders` — **note:** `olist.silver` exists but is empty as of Nov 2026; all Olist data currently lives in bronze.

Coverage: Sep 2016 – Oct 2018 (public Olist dataset snapshot), 99,441 orders.

Key questions:
* How many orders are delivered late vs the estimated delivery date?
* How is the delivery delay distributed, and does it vary by purchase month?

In [ ]:
# C1 — Table overview: rows, uniqueness, date coverage, nulls
profile_df = spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(DISTINCT customer_id) AS distinct_customers,
  MIN(order_purchase_timestamp) AS earliest_purchase,
  MAX(order_purchase_timestamp) AS latest_purchase,
  COUNT_IF(order_approved_at IS NULL) AS null_approved,
  COUNT_IF(order_delivered_carrier_date IS NULL) AS null_carrier_date,
  COUNT_IF(order_delivered_customer_date IS NULL) AS null_delivered_customer,
  COUNT_IF(order_estimated_delivery_date IS NULL) AS null_estimated
FROM olist.bronze.orders
""")
display(profile_df)

In [ ]:
# C2 — Order status distribution
status_df = spark.sql("""
SELECT order_status, COUNT(*) AS orders,
  ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM olist.bronze.orders), 2) AS pct
FROM olist.bronze.orders
GROUP BY order_status
ORDER BY orders DESC
""")
display(status_df)

In [ ]:
# C3 — Delivery delay stats (delivered orders): actual vs estimated
# delay_days > 0 means the order arrived AFTER the estimated date (late)
delay_stats_df = spark.sql("""
WITH d AS (
  SELECT
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
    DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS leadtime_days
  FROM olist.bronze.orders
  WHERE order_status = 'delivered'
)
SELECT
  COUNT(*) AS delivered_orders,
  COUNT_IF(delay_days > 0) AS late_orders,
  ROUND(100.0 * COUNT_IF(delay_days > 0) / COUNT(*), 2) AS pct_late,
  ROUND(AVG(delay_days), 2) AS avg_delay_days,
  ROUND(percentile(delay_days, 0.5), 2) AS median_delay_days,
  MIN(delay_days) AS min_delay,
  MAX(delay_days) AS max_delay,
  ROUND(AVG(leadtime_days), 2) AS avg_leadtime_days,
  COUNT_IF(delay_days > 15) AS very_late_over_15d
FROM d
""")
display(delay_stats_df)

In [ ]:
# C4 — Delay distribution buckets
delay_bucket_df = spark.sql("""
WITH d AS (
  SELECT DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days
  FROM olist.bronze.orders
  WHERE order_status = 'delivered'
)
SELECT
  CASE
    WHEN delay_days <= -20 THEN 'very early (<= -20d)'
    WHEN delay_days BETWEEN -19 AND -10 THEN 'early (-19 to -10d)'
    WHEN delay_days BETWEEN -9 AND -1 THEN 'early (-9 to -1d)'
    WHEN delay_days = 0 THEN 'on time (0d)'
    WHEN delay_days BETWEEN 1 AND 5 THEN 'late (1-5d)'
    WHEN delay_days BETWEEN 6 AND 15 THEN 'late (6-15d)'
    WHEN delay_days > 15 THEN 'very late (>15d)'
  END AS delay_bucket,
  MIN(delay_days) AS sort_key,
  COUNT(*) AS orders,
  ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM d
GROUP BY 1
ORDER BY sort_key
""")
display(delay_bucket_df)

In [ ]:
# C5 — Monthly late-delivery trend
monthly_df = spark.sql("""
WITH d AS (
  SELECT
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delay_days,
    date_trunc('month', order_purchase_timestamp) AS purchase_month
  FROM olist.bronze.orders
  WHERE order_status = 'delivered'
)
SELECT purchase_month,
  COUNT(*) AS delivered,
  COUNT_IF(delay_days > 0) AS late,
  ROUND(100.0 * COUNT_IF(delay_days > 0) / COUNT(*), 1) AS pct_late,
  ROUND(AVG(delay_days), 1) AS avg_delay_days
FROM d
GROUP BY purchase_month
ORDER BY purchase_month
""")
display(monthly_df)

## Findings (executed Nov 2026)

* **99,441 orders**, all unique (`order_id` and `customer_id` both 1:1) — no duplicates to clean.
* **97% delivered** (96,478); `shipped` 1.1%, `canceled` 0.6%, `unavailable` 0.6%, remainder in-flight statuses. All delivered orders have a delivery date (the 2,965 null `delivered_customer_date` rows are non-delivered statuses only).
* **6.77% of delivered orders arrive late** (6,534 / 96,478) vs the estimated date. Olist strongly pads estimates: median delay is **-12 days** (early), average -11.9d, range -147d early to +188d late.
* Distribution: 91.9% early, 1.3% exactly on time, 6.8% late; 1.3% very late (>15d) and 17% arrive over 20 days early — estimates are systematically conservative.
* **Monthly spikes:** late rate jumps to 12.4% (Nov 2017), 14.1% (Feb 2018), **19.0% (Mar 2018)** vs ~3% baseline in 2017 H1 — Black Friday and Q1-2018 fulfillment problems. June 2018 is best (1.2%).
* Modeling implication: class imbalance (~7% late) needs attention; purchase month/seasonality is a strong predictor candidate.